# Retrain on modern data: PERSUADE student essays + current Claude and Gemini essays

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/retrain_modern.ipynb)

Essays written entirely by current models scored 0% AI, because the corpus only had
GPT-3.5/GPT-4-era AI text and no real student essays. This notebook retrains every model on a
corpus that adds both:

* **PERSUADE 2.0**: real essays by US students in grades 6-12 on 15 assignments (CC BY 4.0),
  capped at `PERSUADE_MAX_ESSAYS` (6,000) essays, equally many per assignment;
* **fresh AI essays**, shipped in `data/ai_essays/`: 1,000 by Claude (`claude-opus-5-5`,
  `claude-sonnet-5-5`, `claude-haiku-5-5`) on the same 15 assignments plus college essays, and
  999 by Gemini Flash / Flash-Lite models on 27 school topics.

The ~1,300 AI training essays are **oversampled** (each repeated `OVERSAMPLE` times
during training) so they are not drowned out by the 176k older texts. **Haiku is held out**:
none of its essays is trained on, so its recall shows how the detector does on a Claude model
it has never seen (`configs/splits_modern.json`).

Nothing old is overwritten: the corpus goes to `My Drive/skolfie/data/processed_modern/`,
models to `My Drive/skolfie/models_modern/`, reports to `My Drive/skolfie/reports_modern/`.
The feature cache (`My Drive/skolfie/cache/`) is shared, so GPT-2 only runs on the new texts.

**Binoculars features.** The feature model also gets five `bino_*` features from a modern pair of
language models (Binoculars, Hans et al. 2024, ICML): an *observer* (`Qwen/Qwen2.5-1.5B`) and a
*performer* (`Qwen/Qwen2.5-1.5B-Instruct`, same tokenizer). The Binoculars score is the
observer's perplexity divided by how surprised the observer is by the performer's guesses;
the paper reports strong zero-shot detection of current chat models with it. GPT-2 small
(2019) and its features are kept alongside.
The two models are ~10x slower than GPT-2, so this feature model trains on
`FEATURES_MAX_TRAIN` (40,000) texts and scores at most `FEATURES_MAX_EVAL` per test split; the
PERSUADE and Claude/Gemini essays are always kept (`KEEP_SOURCES`).
`BINO_OBSERVER = ""` turns it off (then everything is trained on, as before).

**Use a GPU runtime** (Runtime → Change runtime type → T4 GPU), then **Runtime → Run all** and
keep the tab open.

| step | what | time on a T4 |
|---|---|---|
| 1-2 | settings, mount Drive, clone, install | ~3 min |
| 3 | unpack the AI essays | <1 min |
| 4 | download M4GT (Drive folder) | ~2-5 min |
| 5 | build the corpus (downloads HC3, AuTexTification, PERSUADE) | ~5-10 min |
| 6 | make the splits | ~1 min |
| 7 | feature model: GPT-2 (cached except for the new texts) + Binoculars pair | ~1-2 h |
| 8 | transformer (DistilBERT + features, 1 epoch) | ~60-80 min |
| 9 | optional: chunk-trained transformer + Mixed rules (`TRAIN_CHUNKS`) | ~45-60 min |
| 10 | stacked model and verdict cut-offs (CPU) | ~3-5 min |
| 11 | summary (plus the old model on the same essays, `COMPARE_OLD`) | ~2-5 min |

About **2.5-4 hours** without the chunk model, 3.5-5 hours with it (`BINO_OBSERVER = ""`:
about 1 hour less).

Step 7 in detail: the two 1.5B models in fp16 on a T4 at 256 tokens take roughly
**0.05-0.1 s per text** (an estimate, not measured here), so ~40,000 train texts plus up to
~25,000 evaluation texts is about 1-2 hours the first time (plus ~5 min to download ~6 GB of
weights). The features are cached on Drive, so a rerun on the same texts takes minutes. The
commented-out Qwen3.8-27B alternative in the settings needs Colab Pro (L4 or A100) and is much
slower: about 0.3-1 s per text in 4-bit on an L4, so even with 12,000 train and 3,000 texts per
split it takes roughly 2.5-7.5 hours.

## 1. Settings

In [ ]:
DRIVE = "/content/drive/MyDrive/skolfie"
FRESH_DRIVE_DIR = f"{DRIVE}/data/raw/fresh"   # optional; empty = repo essays
DATA_DIR = f"{DRIVE}/data/processed_modern"   # new corpus + splits (the old ones stay)
MODELS_DIR = f"{DRIVE}/models_modern"         # new models (the old ones stay in models/)
REPORTS_DIR = f"{DRIVE}/reports_modern"
CACHE_DIR = f"{DRIVE}/cache"                  # shared feature cache (keyed by text)
OLD_MODELS_DIR = f"{DRIVE}/models"            # only read, for the before/after comparison

REPO = "mohaaa825/Skolfie"
BRANCH = "main"
M4GT_FOLDER = "https://drive.google.com/drive/folders/1xC-n5hHiXmaCLmA0q3p8Cbyd7Dk0dpDE"

# Corpus
SOURCES = "hc3,m4gt,autextification,persuade,fresh"
PERSUADE_MAX_ESSAYS = 6000   # human student essays, equally many per prompt; 0 = all 25k
KEEP_RAW_IN_DRIVE = False    # True: keep downloads in My Drive/skolfie/data/raw for reruns
SPLITS_CONFIG = "configs/splits_modern.json"

# Training (same hyperparameters as feature_baseline.ipynb, transformer.ipynb, stack.ipynb)
OVERSAMPLE = "fresh=3"       # repeat each fresh (Claude/Gemini) training essay 3 times
LM_BATCH_SIZE = 32           # GPT-2 feature batch; lower to 16 on out-of-memory
# Binoculars features for the feature model (an observer + performer LM pair with the same
# tokenizer; fits a free T4 in fp16, ~6 GB). BINO_OBSERVER = "" turns them off; an empty
# BINO_PERFORMER gives observer-only features (no cross-perplexity / Binoculars score).
BINO_OBSERVER = "Qwen/Qwen2.5-1.5B"
BINO_PERFORMER = "Qwen/Qwen2.5-1.5B-Instruct"
BINO_QUANTIZATION = None     # None = fp16; "4bit" / "8bit" need bitsandbytes (installed below)
BINO_BATCH_SIZE = 8          # lower to 4 on out-of-memory
FEATURES_MAX_TRAIN = 40000   # train texts for the feature model (only when Binoculars is on)
FEATURES_MAX_EVAL = 5000     # texts per evaluation split (only when Binoculars is on)
KEEP_SOURCES = "fresh,persuade"  # never sampled away by the two limits above
# Qwen3.8-27B (Colab Pro L4/A100): BINO_OBSERVER = "Qwen/Qwen3.8-27B"; BINO_PERFORMER = "";
# BINO_QUANTIZATION = "4bit"; FEATURES_MAX_TRAIN = 12000; FEATURES_MAX_EVAL = 3000
EPOCHS = 1
BATCH_SIZE = 16              # transformer batch; lower to 8 on out-of-memory
MAX_FPR = 0.01               # at most 1% of human validation texts may be called AI
MAX_MISS = 0.05              # at most 5% of AI validation texts may be called Human
TRAIN_CHUNKS = False         # True: also retrain transformer-chunks (Mixed + highlights)
MAX_FALSE_MIXED = 0.05

# Summary
CLAUDE_IN_DIST = ["claude-opus-5-5", "claude-sonnet-5-5",    # trained on: their test rows
                  "gemini-3.1-flash-lite", "gemini-3.5-flash-lite", "gemini-flash-lite-latest"]
CLAUDE_HELD_OUT = ["claude-haiku-5-5"]                       # never trained on
COMPARE_OLD = True           # also score the same essays with the old stacked model

## 2. Check the GPU, mount Drive, clone and install

In [ ]:
import os
import subprocess
import sys

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU: switch to a T4 GPU runtime (Runtime -> Change runtime type).")

from google.colab import drive

drive.mount("/content/drive")


def bino_memory_gb():
    """Rough GPU memory the Binoculars models need: weights (from the "<N>B" in the model ids)
    plus ~4 GB for activations and the logits (vocabulary ~152k)."""
    import re

    per_param = {None: 2.0, "8bit": 1.1, "4bit": 0.65}[BINO_QUANTIZATION or None]
    total = 4.0
    for name in (BINO_OBSERVER, BINO_PERFORMER):
        size = re.search(r"(\d+(?:\.\d+)?)B\b", name or "")
        total += float(size.group(1)) * per_param if size else 0.0
    return total


if BINO_OBSERVER:
    need = bino_memory_gb()
    have = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Binoculars: {BINO_OBSERVER}" + (f" + {BINO_PERFORMER}" if BINO_PERFORMER else "")
          + f" ({BINO_QUANTIZATION or 'fp16'}), needs ~{need:.0f} GB, GPU has {have:.0f} GB")
    if have < need:
        print(f"WARNING: these models will probably not fit in {have:.0f} GB of GPU memory. "
              "Pick a bigger GPU (L4 or A100: Runtime -> Change runtime type, needs Colab "
              "Pro), or use the free-T4 pair Qwen/Qwen2.5-1.5B + Qwen/Qwen2.5-1.5B-Instruct.")

assert DATA_DIR.rstrip("/") != f"{DRIVE}/data/processed", "DATA_DIR would overwrite the old corpus"
assert MODELS_DIR.rstrip("/") != OLD_MODELS_DIR.rstrip("/"), "MODELS_DIR would overwrite old models"
for d in (DATA_DIR, MODELS_DIR, REPORTS_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)
RAW_DIR = f"{DRIVE}/data/raw" if KEEP_RAW_IN_DRIVE else "/content/skolfie_raw"
os.makedirs(RAW_DIR, exist_ok=True)

def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


def git(*args, check=True):
    """Run git; a GITHUB_TOKEN secret reaches it through a credential helper that reads it from
    the environment, so the token never appears in a URL, a command line, an error message or
    .git/config."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    auth = []
    token = secret("GITHUB_TOKEN")
    if token:
        env["SKOLFIE_GITHUB_TOKEN"] = token
        auth = ["-c", "credential.helper=", "-c",
                "credential.helper=!f() { test \"$1\" = get && echo username=x-access-token "
                "&& echo \"password=$SKOLFIE_GITHUB_TOKEN\"; }; f"]
    return subprocess.run(["git", *auth, *args], env=env, check=check)


url = f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    # Clones made by older versions of this notebook kept the token in the remote URL.
    git("-C", "/content/Skolfie", "remote", "set-url", "origin", url)
    git("-C", "/content/Skolfie", "pull", "--ff-only")
else:
    git("clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie")

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e ".[data,features]"
extra = []
if BINO_OBSERVER and BINO_QUANTIZATION:
    extra.append("bitsandbytes")       # 4-bit / 8-bit loading
if "Qwen3.8" in (BINO_OBSERVER or ""):
    extra.append("transformers")       # a new model family needs a recent transformers
if extra:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *extra], check=True)

hf_token = secret("HF_TOKEN")
env = dict(os.environ, **({"HF_TOKEN": hf_token} if hf_token else {}))


def run(cmd):
    """Run a command and stream its output into the notebook as it happens."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env=env)
    for line in proc.stdout:
        sys.stdout.write(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")

## 3. Unpack the AI essays

The 2,000 Claude and Gemini essays ship with the repo (`data/ai_essays/`); extra `<generator>.jsonl`
files from `scripts/import_claude_essays.py` can go in `My Drive/skolfie/data/raw/fresh/`,
which then replaces the repo copies. This copies them next to the other raw data and counts the
essays per model.

In [ ]:
import glob
import gzip
import json
import shutil
from collections import Counter

fresh_dir = f"{RAW_DIR}/fresh"
os.makedirs(fresh_dir, exist_ok=True)
fresh_files = sorted(glob.glob(f"{FRESH_DRIVE_DIR}/*.jsonl"))
if not fresh_files:
    # No essays on Drive: use the Claude and Gemini essays that ship with the repo.
    for gz in sorted(glob.glob("data/ai_essays/*.jsonl.gz")):
        out = f"{fresh_dir}/{os.path.basename(gz)[:-3]}"
        with gzip.open(gz, "rb") as src, open(out, "wb") as dst:
            shutil.copyfileobj(src, dst)
        fresh_files.append(out)
    print(f"Using the repo's AI essays ({len(fresh_files)} files)")
assert fresh_files, f"No .jsonl files in {FRESH_DRIVE_DIR} or data/ai_essays/."
per_generator = Counter()
for path in fresh_files:
    if os.path.dirname(path) != fresh_dir:
        shutil.copy(path, fresh_dir)
    with open(path, encoding="utf-8") as fh:
        per_generator.update(json.loads(line)["generator"] for line in fh if line.strip())
for generator, n in sorted(per_generator.items()):
    print(f"{n:6,}  {generator}")
missing = [g for g in CLAUDE_IN_DIST + CLAUDE_HELD_OUT if g not in per_generator]
if missing:
    print("WARNING: no essays from", ", ".join(missing))

## 4. Download M4GT from Google Drive

As in `build_corpus.ipynb`: only the English subtask-A file is used. PERSUADE is downloaded by
its loader in the next step (also from Google Drive, with `gdown`).

In [ ]:
m4gt_dir = f"{RAW_DIR}/m4gt"
target = f"{m4gt_dir}/SubtaskA.jsonl"
os.makedirs(m4gt_dir, exist_ok=True)

if os.path.exists(target):
    print("M4GT already present:", target)
else:
    download_dir = "/content/m4gt_download"
    !gdown --folder "{M4GT_FOLDER}" -O "{download_dir}" --remaining-ok
    found = sorted(glob.glob(f"{download_dir}/**/*.jsonl", recursive=True))
    subtask_a = [p for p in found if os.path.basename(p).lower() == "subtaska.jsonl"]
    assert subtask_a, "SubtaskA.jsonl not found in the M4GT folder; re-run this cell later."
    shutil.copy(subtask_a[0], target)
    print("copied", subtask_a[0], "->", target)

## 5. Build the new corpus

Into `DATA_DIR` (the old `data/processed/` is left alone). `--strict`: if any source fails
(e.g. Google Drive refuses the PERSUADE download for a while), the build stops instead of
silently training without it. If PERSUADE keeps failing, download the train csv from
https://github.com/scrosseye/persuade_corpus_2.0 by hand and put it (or its .zip) in
`RAW_DIR/persuade/`.

In [ ]:
import pandas as pd

run(["python", "-u", "scripts/build_corpus.py", "--sources", SOURCES,
     "--raw-dir", RAW_DIR, "--out-dir", DATA_DIR,
     "--persuade-max-essays", str(PERSUADE_MAX_ESSAYS), "--strict", "-q"])

corpus = pd.read_parquet(f"{DATA_DIR}/corpus.parquet",
                         columns=["id", "label", "generator", "source", "domain", "n_words"])
display(corpus.groupby(["source", "label"]).size().rename("texts").to_frame())
new = corpus[corpus["source"].isin(["persuade", "fresh"])]
display(new.groupby(["source", "generator"])["n_words"].describe()[["count", "mean", "min",
                                                                    "max"]].round(0))

## 6. Make the splits

`configs/splits_modern.json`: the old held-out GPT-4, Cohere and news, plus `claude-haiku-5-5`
in a family of its own. Opus and Sonnet essays split 80/10/10 like everything else. Essays on
one PERSUADE assignment and grade stay on one side; a Haiku essay on such a topic does not pull
the Opus/Sonnet essays on it out of training (the topic is shared with human essays in train,
so it is no shortcut), but on an AI-only college topic it does.

In [ ]:
run(["python", "-u", "scripts/make_splits.py", "--corpus", f"{DATA_DIR}/corpus.parquet",
     "--config", SPLITS_CONFIG, "--out-dir", DATA_DIR])

splits = pd.read_parquet(f"{DATA_DIR}/splits.parquet")
m = splits.merge(corpus[["id", "generator", "source"]], on="id")
modern = m[m["source"].isin(["fresh", "persuade"])]
display(pd.crosstab(modern["source"] + " / " + modern["generator"], modern["split"]))
n_fresh_train = int(((m["split"] == "train") & (m["source"] == "fresh")).sum())
n_persuade_train = int(((m["split"] == "train") & (m["source"] == "persuade")).sum())
factor = int(OVERSAMPLE.split("=")[1]) if OVERSAMPLE.startswith("fresh=") else 1
print(f"train: {n_fresh_train} Claude essays (x{factor} = {n_fresh_train * factor}) vs "
      f"{n_persuade_train} PERSUADE student essays")

## 7. Feature model (style + GPT-2 + Binoculars + Delta into LightGBM)

Same settings as `feature_baseline.ipynb`, plus `--oversample` and, with `BINO_OBSERVER` set, the
Binoculars features on `FEATURES_MAX_TRAIN` train texts and at most `FEATURES_MAX_EVAL` texts per
evaluation split, always keeping every `KEEP_SOURCES` essay. The stacker in step 10 only uses the
texts that every model scored, so it is then fitted on that sample of `val`.

In [ ]:
oversample = ["--oversample", OVERSAMPLE] if OVERSAMPLE else []
common = ["--corpus", f"{DATA_DIR}/corpus.parquet", "--splits", f"{DATA_DIR}/splits.parquet",
          "--cache-dir", CACHE_DIR, "--save-predictions", *oversample]

bino = []
if BINO_OBSERVER:
    bino = ["--bino-observer", BINO_OBSERVER, "--bino-batch-size", str(BINO_BATCH_SIZE),
            "--max-train", str(FEATURES_MAX_TRAIN), "--keep-sources", KEEP_SOURCES]
    if BINO_PERFORMER:
        bino += ["--bino-performer", BINO_PERFORMER]
    if BINO_QUANTIZATION:
        bino += ["--bino-quantization", BINO_QUANTIZATION]
    if FEATURES_MAX_EVAL:
        bino += ["--max-eval", str(FEATURES_MAX_EVAL)]

run(["python", "-u", "scripts/evaluate.py", *common, "--model", "features",
     "--lm-batch-size", str(LM_BATCH_SIZE), *bino,
     "--out-dir", f"{REPORTS_DIR}/features", "--save-model", f"{MODELS_DIR}/features.joblib"])

## 8. Transformer (DistilBERT + features)

Same settings as `transformer.ipynb`, plus `--oversample`. The longest step: keep the tab open.

In [ ]:
run(["python", "-u", "scripts/evaluate.py", *common, "--model", "transformer",
     "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
     "--out-dir", f"{REPORTS_DIR}/transformer",
     "--save-model", f"{MODELS_DIR}/transformer.joblib"])

## 9. Optional: chunk-trained transformer and Mixed rules

Only with `TRAIN_CHUNKS = True` (as in `chunks.ipynb`). Without it the app uses
`transformer.joblib` for the highlights and the Mixed rule tuned for the old chunk model
(it says so in its notes).

In [ ]:
if TRAIN_CHUNKS:
    run(["python", "-u", "scripts/evaluate.py", *common, "--model", "transformer-chunks",
         "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
         "--out-dir", f"{REPORTS_DIR}/transformer-chunks",
         "--save-model", f"{MODELS_DIR}/transformer-chunks.joblib"])
    run(["python", "-u", "scripts/eval_mixed.py",
         "--corpus", f"{DATA_DIR}/corpus.parquet", "--splits", f"{DATA_DIR}/splits.parquet",
         "--model", f"{MODELS_DIR}/transformer-chunks.joblib",
         "--max-false-mixed", str(MAX_FALSE_MIXED),
         "--out-dir", f"{REPORTS_DIR}/mixed-transformer-chunks"])
else:
    print("Skipped (TRAIN_CHUNKS = False).")

## 10. Stacked model and verdict cut-offs

Same as `stack.ipynb`: fitted on the validation split as it is (not oversampled, so the
probabilities stay calibrated to real data); **AI** only where at most `MAX_FPR` of human
validation texts score, **Human** only where at most `MAX_MISS` of AI validation texts score.

In [ ]:
cmd = ["python", "-u", "scripts/stack.py",
       "--corpus", f"{DATA_DIR}/corpus.parquet", "--splits", f"{DATA_DIR}/splits.parquet",
       "--out-dir", f"{REPORTS_DIR}/stacked",
       "--max-fpr", str(MAX_FPR), "--max-miss", str(MAX_MISS),
       "--save-predictions", "--save-model", f"{MODELS_DIR}/stacked.joblib"]
for member in ("features", "transformer"):
    cmd += ["--member", f"{member}={REPORTS_DIR}/{member}/predictions.parquet",
            "--model-path", f"{member}={MODELS_DIR}/{member}.joblib"]
run(cmd)

## 11. Summary

* **Claude recall** at the stacked model's AI cut-off: Opus/Sonnet on the `test` split
  (models it trained on, essays it did not) vs held-out Haiku on `test_ood_generator`
  (a model it never saw). "AI" is the share called AI, "not Human" the share called AI or
  Uncertain.
* **False positives on real student essays**: PERSUADE human essays in `test`.
* The usual test metrics of the three models.
* With `COMPARE_OLD`, the old stacked model (`models/stacked.joblib`) on the same essays.

In [ ]:
import numpy as np

with open(f"{REPORTS_DIR}/stacked/report.json") as fh:
    stacked_report = json.load(fh)
cutoffs = stacked_report["verdicts"]["cutoffs"]
print(f"cut-offs: AI above {cutoffs['ai_threshold']:.4f}, Human below "
      f"{cutoffs['human_threshold']:.4f}")

meta = pd.read_parquet(f"{DATA_DIR}/corpus.parquet",
                       columns=["id", "text", "generator", "source", "label"])
preds = pd.read_parquet(f"{REPORTS_DIR}/stacked/predictions.parquet").merge(
    meta[["id", "generator", "source"]], on="id")


def rates(p, ai_cut, human_cut):
    p = np.asarray(p)
    return {"n": len(p), "AI": float((p > ai_cut).mean()) if len(p) else float("nan"),
            "not Human": float((p >= human_cut).mean()) if len(p) else float("nan")}


groups = [(g, "test", "trained-on model") for g in CLAUDE_IN_DIST]
groups += [(g, "test_ood_generator", "held out") for g in CLAUDE_HELD_OUT]
selections = {}
rows = []
for generator, split, kind in groups:
    sel = preds[(preds["split"] == split) & (preds["generator"] == generator)]
    selections[generator] = sel["id"].tolist()
    rows.append({"texts": generator, "split": split, "kind": kind,
                 **rates(sel["p_ai"], cutoffs["ai_threshold"], cutoffs["human_threshold"])})
human = preds[(preds["split"] == "test") & (preds["source"] == "persuade")]
selections["persuade (human)"] = human["id"].tolist()
r = rates(human["p_ai"], cutoffs["ai_threshold"], cutoffs["human_threshold"])
rows.append({"texts": "persuade (human)", "split": "test", "kind": "false positives",
             "n": r["n"], "AI": r["AI"], "not Human": r["not Human"]})
summary = pd.DataFrame(rows).set_index("texts")
print("\nStacked model at its cut-offs (for PERSUADE, 'AI' is the false-positive rate):")
display(summary.style.format({"AI": "{:.1%}", "not Human": "{:.1%}"}))

metric_rows = []
for name in ("features", "transformer", "stacked"):
    with open(f"{REPORTS_DIR}/{name}/report.json") as fh:
        rep = json.load(fh)
    for split, res in rep["results"].items():
        m = res["metrics"]
        metric_rows.append({"model": name, "split": split, "n": m["n"], "AUROC": m["auroc"],
                            "F1": m["f1"], "FPR@95%TPR": m["fpr_at_95tpr"], "ECE": m["ece"]})
metrics = pd.DataFrame(metric_rows).set_index(["model", "split"])
print("\nUsual metrics:")
display(metrics.round(4))
verdict_rows = {split: {**{f"human->{k}": v for k, v in t["human_texts"].items() if k != "n"},
                        **{f"ai->{k}": v for k, v in t["ai_texts"].items() if k != "n"}}
                for split, t in stacked_report["verdicts"]["splits"].items()}
print("\nStacked verdict shares per split:")
display(pd.DataFrame(verdict_rows).T.round(3))

In [ ]:
old_summary = None
old_path = f"{OLD_MODELS_DIR}/stacked.joblib"
if COMPARE_OLD and os.path.exists(old_path):
    import joblib

    old = joblib.load(old_path)
    texts = meta.set_index("id")["text"]
    old_rows = []
    for name, ids in selections.items():
        p = old.predict_proba(texts.loc[ids].tolist()) if ids else np.zeros(0)
        r = rates(p, old.cutoffs["ai_threshold"], old.cutoffs["human_threshold"])
        old_rows.append({"texts": name, "old: AI": r["AI"], "old: not Human": r["not Human"]})
    old_summary = pd.DataFrame(old_rows).set_index("texts")
    both = summary[["n", "AI", "not Human"]].join(old_summary)
    print("New vs old stacked model on the same test essays:")
    display(both.style.format({c: "{:.1%}" for c in both.columns if c != "n"}))
elif COMPARE_OLD:
    print(f"No old model at {old_path}; skipped the comparison.")


def as_md(df):
    try:
        return df.to_markdown()
    except ImportError:  # pandas needs the tabulate package for to_markdown
        return "```\n" + df.to_string() + "\n```"


with open(f"{REPORTS_DIR}/modern_summary.md", "w") as fh:
    fh.write("# Modern-data retrain\n\n")
    fh.write(f"cut-offs: {json.dumps(cutoffs)}\n\n")
    fh.write(as_md(summary if old_summary is None else summary.join(old_summary)))
    fh.write("\n\n" + as_md(metrics.round(4)) + "\n")
print(f"Saved {REPORTS_DIR}/modern_summary.md")

To use the new models, point the app at them: in `app.ipynb` set
`MODELS_DIR = "/content/drive/MyDrive/skolfie/models_modern"` and
`REPORTS_DIR = "/content/drive/MyDrive/skolfie/reports_modern"`. The fairness check
(`fairness.ipynb`) has not been rerun for them, so the app shows no known limitations until it
is (point its folders at the new ones too). Tell Claude in the project thread when this has
finished; it reads the reports from Drive.

With Binoculars on, the app's feature model loads the two Qwen models on the first check, so the
app needs a GPU runtime with enough memory too (about 6 GB for the 1.5B pair).